# Conv-LSTM Predictor — First Checkpoint Training (SP-1.1)

Open this notebook in **Colab → Runtime → Change runtime type → T4 GPU** (free tier is enough).

This notebook:
1. Clones the trading-radar repo (you'll be prompted for a GitHub Personal Access Token if the repo is private)
2. Installs Python deps (torch is already in the Colab base image)
3. Fetches BTC/USDT 1h history from Binance public klines API (~30 sec)
4. Trains the Conv-LSTM defined in `backend/app/ml/model.py` (~30-60 min on T4)
5. Evaluates on the 5 frozen regime windows from `app.ml.regimes`
6. Downloads the resulting `.pt` + `eval.json` to your laptop

**Acceptance gate:** MAE ≤ 1.5% on ALL 5 regimes (single-window failure rejects activation per SP-1 spec sec 2 row 13).

After download, scp the `.pt` to the Hetzner server and run `tools/ml/register.py` to activate it. See `tools/ml/README.md` for the post-training steps.

## 1. Clone the repo

If the repo is **public**, leave the PAT prompt blank and press Enter.
If **private**, create a fine-scoped PAT at https://github.com/settings/personal-access-tokens/new (only `Contents: read` on the trading-radar repo), then paste it below.

In [ ]:
import getpass, os, subprocess

PAT = getpass.getpass('GitHub Personal Access Token (blank if public): ')
REPO = 'naga1412/v5_Trade_bot'
url = f'https://{PAT + "@" if PAT else ""}github.com/{REPO}.git'

if not os.path.exists('/content/v5_Trade_bot'):
    subprocess.check_call(['git', 'clone', '--depth=1', url, '/content/v5_Trade_bot'])
%cd /content/v5_Trade_bot
!git log -1 --oneline

## 2. Install dependencies

Colab already has `torch`, `pandas`, `numpy`. We add `pyarrow` (Parquet) and `requests`.

In [ ]:
!pip install --quiet pyarrow requests

## 3. Fetch BTC/USDT 1h history from Binance

Pulls 2017-08-17 → today, ~70-80k bars, 70-80 paginated requests. Writes to `data/ml/btcusdt_1h.parquet`.

In [ ]:
from datetime import datetime, timezone
today = datetime.now(timezone.utc).strftime('%Y-%m-%d')
!python -m tools.ml.fetch_ohlcv \
    --symbol BTCUSDT --interval 1h \
    --start 2017-08-17 --end {today} \
    --out data/ml/btcusdt_1h.parquet

## 4. Train the Conv-LSTM

Defaults: 30 epochs, Adam lr=3e-4, batch=64, early-stop patience=5. T4 GPU runs each epoch in ~1-3 min depending on dataset size; total wall time typically 30-60 min.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

from datetime import datetime, timezone
VERSION = 'v1-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
OUT_DIR = f'data/ml/runs/{VERSION}'

!python -m tools.ml.train \
    --data data/ml/btcusdt_1h.parquet \
    --out-dir {OUT_DIR} \
    --version-tag {VERSION} \
    --device auto \
    --epochs 30 \
    --patience 5

## 5. Inspect eval results

Pretty-print regime-by-regime MAE, highlight pass/fail per the SP-1 acceptance gate.

In [ ]:
import json, glob, os
eval_files = sorted(glob.glob(f'{OUT_DIR}/eval_*.json'))
with open(eval_files[-1]) as f:
    doc = json.load(f)

print(f"version: {doc['version']}")
print(f"trained_at: {doc['trained_at']}")
print(f"best_val_mae: {doc['training']['best_val_mae']:.4f} (epoch {doc['training']['best_epoch']})")
print(f"acceptance_threshold: {doc['acceptance_threshold']}")
print()
print(f"{'regime':<20} {'mae':>8} {'samples':>8}  result")
print('-' * 52)
for name, r in doc['regime_results'].items():
    flag = '✓ PASS' if r['passes_acceptance'] else '✗ FAIL'
    print(f"{name:<20} {r['mae']:>8.4f} {r['samples']:>8}  {flag}")
print()
print('ALL REGIMES PASS:', doc['all_regimes_pass'])

## 6. Download checkpoint + eval to your laptop

Triggers the browser download dialog. Save both files to a known location, then SCP the `.pt` to the Hetzner server (see `tools/ml/README.md` for the exact commands).

In [ ]:
from google.colab import files
import glob
for f in sorted(glob.glob(f'{OUT_DIR}/conv_lstm_*.pt')) + sorted(glob.glob(f'{OUT_DIR}/eval_*.json')):
    print('downloading', f)
    files.download(f)

## 7. Next steps (run on your laptop)

```powershell
# 1. SCP the .pt to the Hetzner server (replace VERSION with the timestamp from cell 4)
scp -i $HOME\.ssh\oracle_key conv_lstm_VERSION.pt root@95.216.187.204:/opt/trading-radar/backend/data/ml-cache/
scp -i $HOME\.ssh\oracle_key eval_VERSION.json root@95.216.187.204:/tmp/

# 2. SSH in and register the checkpoint via the admin API (force=true for first checkpoint)
ssh -i $HOME\.ssh\oracle_key root@95.216.187.204 "docker exec -it tr-backend python -m tools.ml.register --checkpoint /app/data/ml-cache/conv_lstm_VERSION.pt --eval /tmp/eval_VERSION.json --base-url http://localhost:8000 --activate --force"

# 3. Restart backend so it loads the new checkpoint
ssh -i $HOME\.ssh\oracle_key root@95.216.187.204 "docker compose -f /opt/trading-radar/docker-compose.yml restart backend"
```

After step 3, the dashboard's `predictions.ghost_*` columns start populating on every closed BTC/USDT 1h candle, and the chart's ghost-candle overlay (built in SP-1 Phase E) lights up.